In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from pathlib import Path
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [6]:
####### load common directories and data
time_interval = 10 #sec/frame
basedir = Path('E:/Aaron/Combined_37C_Confocal_PCA_shape/')
datadir = basedir.joinpath('Data_and_Figs')
savedir = basedir.joinpath('Detailed_Balance/')
if not savedir.exists():
    savedir.mkdir()
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)
nbins = len(centers.iloc[:,0])
ttot = time_interval * 180
ntrans = 1
bsiter = 3000


## restrict the treatments and PCs specifically for bootstrapping
bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = []#[[1,2]]
alldatabs = True

In [7]:
#### all the cgps origins determinned by visual inspection

#### SHAPE alignment origins
allorigins = [[[8,8],[8,8],[8,8],[8,7],[8,8],[8,8],[8,8]],
                [[8,8],[8,8],[8,8],[8,8],[8,8],[8,8]],
                    [[8,8],[8,8],[8,8],[8,8],[8,8]],
                        [[8,8],[8,8],[8,8],[8,8]],
                            [[8,8],[8,8],[8,8]],
                                [[8,8],[8,8]],
                                    [[7,7]]]


# #### WIDTH alignment origins
# allorigins = [[[8,8],[8,8],[9,8],[8,8],[9,7],[9,8],[9,8]],
#                 [[8,8],[8,8],[8,8],[8,8],[8,8],[8,8]],
#                     [[8,8],[8,8],[8,8],[8,8],[8,8]],
#                         [[8,8],[8,8],[8,8],[8,8]],
#                             [[8,8],[8,8],[8,8]],
#                                 [[6,8],[8,8]],
#                                     [[8,8]]]

# #### PLANAR alignment origins
# allorigins = [[[7,8],[8,6],[8,9],[8,8],[8,8],[8,8],[9,7]],
#                 [[8,6],[8,9],[8,8],[8,7],[8,7],[8,8]],
#                     [[8,8],[7,8],[8,8],[8,8],[9,8]],
#                         [[9,8],[7,8],[8,8],[8,8]],
#                             [[8,8],[8,8],[8,8]],
#                                 [[8,7],[8,8]],
#                                     [[8,8]]]

In [4]:
############# create all CGPSs #############

npcs = centers.shape[1]
for a in range(1,npcs+1):
    for b in range(1,npcs+1):
        if a == b:
            continue
        elif savedir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
            print('Already made this CGPS')
            continue
        else:
            #set the PCs
            whichpcs = [a,b]
            if __name__ ==  '__main__':
                
                ########### get raw transitions and pairs ###########
                rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                        FullFrame, #pandas dataframe with all of the cgps binned data
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        savedir, #where to save the aggregated trajectories
                        )
                
                ########### interpolate all transitions so that only individual transitions are made ###########
                transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                        rawtrans, #pandas dataframe with all of the cgps binned data
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        savedir, #where to save the aggregated trajectories
                        )

                ############## get the counts of cells leaving 
                trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                        transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        savedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        )
                
                
            

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2852.4975055084733 minutes
Total time observed in this CGPS was 6821.005417985739 minutes
Total time observed in this CGPS was 1531.225254110008 minutes
Total time observed in this CGPS was 3474.1716518898156 minutes
Total time observed in this CGPS was 32.047656388510376 minutes
Total time observed in this CGPS was 1446.607706349432 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2830.3856398711696 minutes
Total time observed in this CGPS was 6808.984050738319 minutes
Total time observed in this CGPS was 1530.4463710380196 minutes
Total time observed in this CGPS was 3498.381772907427 minutes
Total time observed in this CGPS was 33.60680840922348 minutes
Total time observed in this CGPS was 1436.3095298891096 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajec

Total time observed in this CGPS was 6619.165626783678 minutes
Total time observed in this CGPS was 1517.4025276699879 minutes
Total time observed in this CGPS was 3381.128704492935 minutes
Total time observed in this CGPS was 34.69424819188624 minutes
Total time observed in this CGPS was 1414.2905006767692 minutes
Finished finding transition rates
Already made this CGPS
Already made this CGPS
Already made this CGPS
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2766.5421785276553 minutes
Total time observed in this CGPS was 6715.763373880998 minutes
Total time observed in this CGPS was 1519.3592868387384 minutes
Total time observed in this CGPS was 3421.2724694423146 minutes
Total time observed in this CGPS was 35.43602495669529 minutes
Total time observed in this CGPS was 1423.209745764328 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2806.66549

In [8]:
########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############

npcs = centers.shape[1]
for a in range(1,npcs+1):
    for b in range(1,npcs+1):
        #set the PCs
        whichpcs = [a,b]
        #pass if we want to restrict PCs
        if len(bspcs)>0 and whichpcs not in bspcs:
            continue
        ## use a separate savedir to bootstrap using all data
        if alldatabs:
            bssavedir = savedir.joinpath('alldatabs')
            if not bssavedir.exists():
                bssavedir.mkdir()
        else:
            bssavedir = savedir
            
        if a == b:
            continue
        elif bssavedir.joinpath(f'PC{b}-PC{a}_bootstrapped_{ntrans}_transitions.csv').exists():
            print('Already made this plot')
            continue
        else:
            if __name__ ==  '__main__':
                #### open the transitions
                rawtrans = pd.read_csv(savedir.joinpath(
                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)
                
                #merge all treatments if bootstrapping with all data
                if alldatabs:
                    rawtrans.loc[:,'Treatment'] = 'alldata'
                    
                #restrict to bootstrapped treatments if desired
                if len(bstreats)>0:
                    rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]
                
                
                ############## BOOTSTRAP MANY TRAJECTORIES ##########
                bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                        rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        bssavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ttot, #set the total bootstrap time
                        ntrans, #how many transitions to sample at each step
                        bsiter, #number of times to bootstrap
                        )


                ############# open average bootstrapped currents ###################
                bsfield_sep = DetailedBalance.get_avg_current_error(
                        bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        bssavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ntrans, #how many transitions to sample at each step
                        )
    
                
                
                ############# measure aer and cycling frequencies ###########
                #add specific scaling
                xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
                #set the origin to the actual center
                center = allorigins[int(a-1)][int(b-(2+a-1))]

                DetailedBalance.get_aer_cf(
                    bstrans,
                    nbins, #how many bins in the x and y cgps axes
                    xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                    center, #origin in [x bin,y bin]
                    bssavedir, #where to save calculated aers and cfs
                    whichpcs, #which two PCs to use in the cgps [x,y]
                    ntrans, #how many transitions to sample at each step
                    )
                
                

                ############### measure aer and cycling frequency for the raw transitions
                #get the area scaling in x and y based on the size of the bins in the cgps
                results = []
                for i, cell in rawtrans.groupby('CellID'):
                    #sort data and get continuous transitions in order
                    cell, runs = utils.get_consecutive_transitions(cell)
                    for r in runs:
                        c = cell.iloc[r].reset_index(drop=True)
                        results.append(DetailedBalance.get_area_enclosing_rate((
                            c,
                            nbins,
                            xyscaling,
                            center,
                            )))

                #make a dataframe and save it
                allaers = pd.concat(results, ignore_index = True)
                allaers.to_csv(bssavedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



Boostrapping trajectories with 1 transition samples for alldata


100%|██████████| 3000/3000 [02:48<00:00, 17.80it/s]


Interpolating trajectories for alldata


100%|██████████| 3000/3000 [00:05<00:00, 536.28it/s] 


Calculating bootstrapped CGPS transition rates for alldata


...

100%|██████████| 3000/3000 [02:43<00:00, 18.30it/s]


Interpolating trajectories for alldata


100%|██████████| 3000/3000 [00:04<00:00, 639.94it/s] 


Calculating bootstrapped CGPS transition rates for alldata


 38%|███▊      | 1154/3000 [00:37<00:53, 34.61it/s]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

